# Dashboard Prep

This notebook prepares reusable outputs for the Streamlit dashboard.


In [1]:
import os
import sys
from pathlib import Path

def find_project_root(start: Path) -> Path:
    current = start.resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "src").exists():
            return candidate
    raise FileNotFoundError("Could not find project root containing 'src/'")

project_root = find_project_root(Path.cwd())

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

os.chdir(project_root)

print("Project root:", project_root)
print("src exists:", (project_root / "src").exists())

Project root: C:\Users\Aarush\OneDrive\Desktop\customer_segmentation_starter\customer-segmentation-starter
src exists: True


In [2]:
from pathlib import Path
import pandas as pd
from src.rfm import build_rfm, add_rfm_scores
from src.segmentation import label_segments
from src.cohort import build_cohort_retention


In [3]:
from pathlib import Path

clean_path = Path('data/processed/online_retail_clean.csv')
if not clean_path.exists():
    raise FileNotFoundError(f'File not found: {clean_path.resolve()}')

out_dir = Path('../data/processed/dashboard')
out_dir.mkdir(parents=True, exist_ok=True)
if not clean_path.exists():
    raise FileNotFoundError('Run 02_preprocessing.ipynb first.')
df = pd.read_csv(clean_path, parse_dates=['InvoiceDate'])


In [4]:
rfm = label_segments(add_rfm_scores(build_rfm(df)))
retention = build_cohort_retention(df)
segment_counts = rfm['Segment'].value_counts().rename_axis('Segment').reset_index(name='CustomerCount')


In [5]:
rfm.to_csv(out_dir / 'rfm_segments.csv', index=False)
segment_counts.to_csv(out_dir / 'segment_counts.csv', index=False)
retention.to_csv(out_dir / 'cohort_retention.csv')
print('Saved dashboard assets to', out_dir)


Saved dashboard assets to ..\data\processed\dashboard


In [6]:
segment_counts


,Segment,CustomerCount
0,Hibernating,1504
1,Loyal Customers,1033
2,Potential Loyalist,665
3,At Risk,646
4,Champions,490


## Output

These CSV files can be loaded directly by the Streamlit app for faster demos.
